# 🦜 Nhân bản giọng nói của bạn — VieNeu-TTS (Google Colab)

**Dành cho người không chuyên kỹ thuật.** Bạn chỉ cần làm 2 việc:
1. Bấm menu **Runtime → Run all** (hoặc Chạy tất cả).
2. Khi được hỏi, **tải lên file ghi âm** của bạn (`.m4a`/`.wav`, dài 5–60 giây).

Notebook sẽ TỰ ĐỘNG: phiên âm giọng bạn → nhân bản giọng → đọc thử một câu tiếng Việt bằng **chính giọng của bạn** và cho bạn tải về.

> 💡 Nên bật GPU miễn phí: **Runtime → Change runtime type → T4 GPU** (không bắt buộc, chỉ nhanh hơn).


## Bước 1 — Cài đặt (tự động, ~2–4 phút)


In [ ]:
!pip -q install vieneu imageio-ffmpeg faster-whisper 2>/dev/null
print('✅ Cài xong.')


## Bước 2 — Tải lên file ghi âm của bạn
Bấm nút **Choose Files** xuất hiện bên dưới và chọn file ghi âm (`Ghi âm của tôi.m4a`...).


In [ ]:
from google.colab import files
import os, imageio_ffmpeg, subprocess
up = files.upload()                       # chọn file ghi âm
src = list(up.keys())[0]
ff = imageio_ffmpeg.get_ffmpeg_exe()
# Chuyển sang wav 24kHz mono + cắt 8 giây sạch đầu tiên làm mẫu
subprocess.run([ff,'-y','-i',src,'-ac','1','-ar','24000','full.wav'], stderr=subprocess.DEVNULL)
subprocess.run([ff,'-y','-i','full.wav','-ss','0','-t','8','reference.wav'], stderr=subprocess.DEVNULL)
print('✅ Đã chuẩn bị mẫu giọng: reference.wav (8 giây đầu)')


## Bước 3 — Tự phiên âm đoạn mẫu (Whisper)
VieNeu cần biết *bạn nói gì* trong đoạn mẫu. Bước này làm tự động — bạn KHÔNG cần gõ gì.


In [ ]:
from faster_whisper import WhisperModel
w = WhisperModel('small', device='cpu', compute_type='int8')
segs, info = w.transcribe('reference.wav', language='vi')
ref_text = ' '.join(s.text.strip() for s in segs).strip()
print('📝 Câu trong đoạn mẫu (tự nhận diện):')
print('   ', ref_text)
# Nếu nhận diện sai, bạn có thể sửa tay tại đây:
# ref_text = 'câu đúng bạn đã nói'


## Bước 4 — Nhân bản giọng & đọc thử
Lần chạy đầu sẽ tải model VieNeu (~vài trăm MB) từ HuggingFace — chờ một chút.


In [ ]:
from vieneu import Vieneu
tts = Vieneu(emotion='natural')          # chế độ chất lượng cao
cau_doc = 'Xin chào các anh chị, đây là giọng nói của tôi được tạo lại bằng trí tuệ nhân tạo.'
audio = tts.infer(text=cau_doc, ref_audio='reference.wav', ref_text=ref_text)
tts.save(audio, 'giong_cua_toi.wav')
print('✅ Đã tạo: giong_cua_toi.wav')
from IPython.display import Audio, display
display(Audio('giong_cua_toi.wav'))      # nghe thử ngay


## Bước 5 — Tải file giọng về máy


In [ ]:
from google.colab import files
files.download('giong_cua_toi.wav')


## (Tuỳ chọn) Bước 6 — Đọc cả bài giảng bằng giọng của bạn
Dán đoạn lời cần đọc vào giữa hai dấu nháy ba. Có thể chạy nhiều lần với nội dung khác nhau.


In [ ]:
noi_dung = '''
Chào mừng các anh chị đến với Chương 4: Soạn thảo và trình bày văn bản.
Trong chương này, chúng ta sẽ tìm hiểu khái niệm, phân loại và thể thức của văn bản.
'''
audio2 = tts.infer(text=noi_dung.strip(), ref_audio='reference.wav', ref_text=ref_text)
tts.save(audio2, 'bai_giang.wav')
from IPython.display import Audio, display
display(Audio('bai_giang.wav'))
from google.colab import files; files.download('bai_giang.wav')
